# Phase 2 — Delta Lake

Phase 1 kết thúc bằng một vết thương hở. Bạn gộp hơn nghìn file nhỏ thành một file to
bằng tay, và ta ghi nhận: **nếu lúc đó có người đang query, họ sẽ thấy dữ liệu hỏng** —
file cũ đã xoá, file mới chưa ghi xong.

Một thư mục Parquet trần thiếu ba thứ, và cả ba đều chí mạng:

| Thiếu gì | Hậu quả thật |
|---|---|
| **Giao dịch** (transaction) | Job ghi nửa chừng thì chết → lake còn lại file rác, không ai biết file nào hợp lệ |
| **Phiên bản** | Ghi đè nhầm là mất sạch. Không có "Ctrl-Z" |
| **Upsert** | Muốn sửa 100 dòng trong 10 triệu dòng → phải đọc và ghi lại toàn bộ |

Delta Lake vá đúng ba lỗ đó. Và cách nó vá thì đơn giản đến mức gây sốc:
**thêm một thư mục `_delta_log/` chứa vài file JSON.** Hết. Dữ liệu vẫn là Parquet
y như Phase 1.

```
  3. Compute engine   → DuckDB / delta-rs      (Phase 3 thay bằng Spark)
  2. Table format     → DELTA LAKE          ← PHASE 2 DỰNG TẦNG NÀY
  1. Object storage   → MinIO                  (xong Phase 0-1)
```

> **Vì sao dùng `delta-rs` chứ không phải `delta-spark`?**
> Spark là nội dung Phase 3 — kéo nó vào đây sẽ trộn hai bài học lớn làm một, và bạn sẽ
> không phân biệt được đâu là *Delta* đâu là *Spark*. `delta-rs` (thư viện Rust, không cần
> JVM) cài đặt **đúng Delta protocol chuẩn**, nên `_delta_log/` sinh ra giống hệt thứ
> Databricks thật ghi. Đến Phase 3, Spark sẽ đọc được thẳng bảng bạn tạo hôm nay —
> đó chính là bằng chứng Delta là **định dạng mở**, không phải API riêng của ai.

---
## Bước 1 — Bảng Delta đầu tiên

Ta lấy 500.000 chuyến taxi làm bảng **bronze** (dữ liệu thô, chưa xử lý — thuật ngữ này
sẽ thành trọng tâm ở Phase 4).

Một điểm khác Phase 1: ta thêm cột `trip_id`. Dữ liệu taxi gốc không có khoá chính,
mà **không có khoá thì không `MERGE` được**. Tự sinh khoá thay thế (*surrogate key*)
là việc data engineer làm hằng ngày.

In [1]:
import os, json, time
import boto3, duckdb, pandas as pd, pyarrow as pa
from deltalake import DeltaTable, write_deltalake

BUCKET = os.environ['LAKEHOUSE_BUCKET']
s3  = boto3.client('s3', endpoint_url=os.environ['AWS_ENDPOINT_URL'])
con = duckdb.connect()
con.sql(f"""
    CREATE OR REPLACE SECRET minio (
        TYPE s3, KEY_ID '{os.environ['AWS_ACCESS_KEY_ID']}',
        SECRET '{os.environ['AWS_SECRET_ACCESS_KEY']}',
        ENDPOINT 'minio:9000', USE_SSL false, URL_STYLE 'path')
""")

# delta-rs không đọc biến môi trường AWS như boto3 — phải đưa tận tay
SO = {
    'AWS_ACCESS_KEY_ID'     : os.environ['AWS_ACCESS_KEY_ID'],
    'AWS_SECRET_ACCESS_KEY' : os.environ['AWS_SECRET_ACCESS_KEY'],
    'AWS_ENDPOINT_URL'      : os.environ['AWS_ENDPOINT_URL'],
    'AWS_REGION'            : 'us-east-1',
    'AWS_ALLOW_HTTP'        : 'true',      # MinIO nội bộ chạy http, không https
}

RAW   = f's3://{BUCKET}/raw/yellow_taxi/*.parquet'
TABLE = f's3://{BUCKET}/phase2/bronze_trips'
PREF  = 'phase2/bronze_trips'              # cùng chỗ đó, nhìn bằng con mắt boto3

bronze = con.sql(f"""
    SELECT row_number() OVER (ORDER BY tpep_pickup_datetime) AS trip_id,
           tpep_pickup_datetime AS pickup_at,
           PULocationID  AS pu_zone,
           DOLocationID  AS do_zone,
           passenger_count,
           trip_distance,
           total_amount
    FROM read_parquet('{RAW}')
    WHERE tpep_pickup_datetime >= '2024-01-01' AND tpep_pickup_datetime < '2024-02-01'
    ORDER BY tpep_pickup_datetime
    LIMIT 500000
""").to_arrow_table()

# Dọn sạch sân chơi phase 2 trước khi bắt đầu.
# Vì sao cần: notebook này đếm SỐ PHIÊN BẢN của bảng. Chạy lại lần hai trên bảng cũ
# thì version sẽ nối tiếp 4, 5, 6... và mọi giải thích bên dưới lệch hết. Chạy lại cho
# ra đúng kết quả cũ — tính chất đó tên là IDEMPOTENCY, và là trọng tâm của Phase 7.
def don_phase2():
    keys, token = [], None
    while True:
        kw = dict(Bucket=BUCKET, Prefix='phase2/')
        if token:
            kw['ContinuationToken'] = token
        r = s3.list_objects_v2(**kw)
        keys += [o['Key'] for o in r.get('Contents', [])]
        if not r.get('IsTruncated'):
            break
        token = r['NextContinuationToken']
    for i in range(0, len(keys), 1000):
        s3.delete_objects(Bucket=BUCKET, Delete={'Objects': [{'Key': k} for k in keys[i:i+1000]]})
    return len(keys)

print(f'Đã dọn {don_phase2()} object cũ')

write_deltalake(TABLE, bronze, mode='overwrite', storage_options=SO)
print(f'✓ Đã tạo bảng Delta với {bronze.num_rows:,} dòng')

Đã dọn 275 object cũ
✓ Đã tạo bảng Delta với 500,000 dòng


In [2]:
# Nhìn xem trên storage thực sự có gì
for k, sz in sorted((o['Key'], o['Size']) for o in s3.list_objects_v2(Bucket=BUCKET, Prefix=PREF)['Contents']):
    print(f'{sz/1e6:8.2f} MB  {k}')

    0.00 MB  phase2/bronze_trips/_delta_log/00000000000000000000.json
    7.45 MB  phase2/bronze_trips/part-00000-1643a0dc-3f75-457a-8d1b-7772d1c8e08e-c000.snappy.parquet


Đọc kỹ danh sách trên. Chỉ có **hai loại thứ**:

1. File `part-*.snappy.parquet` — **y hệt Phase 1**, không có gì mới.
2. Thư mục `_delta_log/` với một file JSON bé tí.

Đó là toàn bộ Delta Lake. Không có server, không có database, không có tiến trình nền
nào đang chạy. **Một bảng Delta chỉ là một thư mục.** Chép thư mục đó sang máy khác là
chép được cả bảng, cả lịch sử.

Đây cũng là câu trả lời cho *"lakehouse khác data warehouse chỗ nào?"*: warehouse giữ
dữ liệu trong định dạng riêng của nó, muốn lấy ra phải đi qua nó. Lakehouse để dữ liệu
nằm trần trên object storage theo chuẩn mở — ai đọc cũng được.

---
## Bước 2 — Mở `_delta_log/` đọc bằng tay

> Đây là **trọng tâm của cả Phase 2**. Nếu chỉ nhớ một cell trong notebook này, hãy nhớ cell dưới.

Hiểu được file JSON này thì Delta hết là hộp đen vĩnh viễn.

In [3]:
def doc_log(version, table_prefix=PREF, tom_tat=True):
    """In nội dung một commit trong _delta_log — mỗi dòng file là một 'action'."""
    key = f'{table_prefix}/_delta_log/{version:020d}.json'
    body = s3.get_object(Bucket=BUCKET, Key=key)['Body'].read().decode()
    print(f'━━━ {key} ━━━')
    for line in body.strip().split('\n'):
        act = json.loads(line)
        loai = list(act.keys())[0]
        noi_dung = act[loai]
        if tom_tat and loai == 'add':
            noi_dung = {k: v for k, v in noi_dung.items() if k != 'stats'}
        print(f'\n▸ action: {loai}')
        print(json.dumps(noi_dung, indent=2, ensure_ascii=False)[:900])

doc_log(0)

━━━ phase2/bronze_trips/_delta_log/00000000000000000000.json ━━━

▸ action: commitInfo
{
  "timestamp": 1787481243668,
  "operation": "WRITE",
  "operationParameters": {
    "mode": "Overwrite"
  },
  "engineInfo": "delta-rs:py-1.6.3",
  "operationMetrics": {
    "num_added_files": 1,
    "num_removed_files": 0,
    "num_partitions": 0,
    "num_added_rows": 500000,
    "execution_time_ms": 98
  },
  "clientVersion": "delta-rs.py-1.6.3"
}

▸ action: protocol
{
  "minReaderVersion": 3,
  "minWriterVersion": 7,
  "readerFeatures": [
    "timestampNtz"
  ],
  "writerFeatures": [
    "timestampNtz"
  ]
}

▸ action: metaData
{
  "id": "882ffc08-4988-42f1-84e1-472f7ea4385f",
  "name": null,
  "description": null,
  "format": {
    "provider": "parquet",
    "options": {}
  },
  "schemaString": "{\"type\":\"struct\",\"fields\":[{\"name\":\"trip_id\",\"type\":\"long\",\"nullable\":true,\"metadata\":{}},{\"name\":\"pickup_at\",\"type\":\"timestamp_ntz\",\"nullable\":true,\"metadata\":{}},{\"nam

### Ba loại action bạn vừa thấy

| Action | Nói lên điều gì |
|---|---|
| `protocol` | "Đọc bảng này cần reader/writer phiên bản mấy" — cơ chế để Delta tiến hoá mà không làm hỏng client cũ |
| `metaData` | Schema của bảng, dạng JSON. **Schema nằm trong log, không nằm trong file Parquet** |
| `add` | "Từ commit này trở đi, file X thuộc về bảng" — kèm kích thước, thời điểm, và **thống kê** |

Điểm mấu chốt, hãy đọc chậm:

> **Bảng = tập hợp các file mà log nói là thuộc về nó.**
> Không phải "mọi file có trong thư mục".

File Parquet nằm trong thư mục nhưng không được `add` nào nhắc tới thì **vô hình** —
đọc bảng sẽ không thấy nó. Đây chính là chỗ mọi phép màu bắt nguồn: muốn thêm/bớt/sửa
dữ liệu, Delta không đụng vào file cũ, nó chỉ **ghi thêm một dòng vào log**.

In [4]:
# Phần bị giấu ở trên: thống kê min/max của từng file — Delta chép sẵn ra log
key  = f'{PREF}/_delta_log/{0:020d}.json'
body = s3.get_object(Bucket=BUCKET, Key=key)['Body'].read().decode()
add  = [json.loads(l)['add'] for l in body.strip().split('\n') if l.startswith('{"add"')][0]
print(json.dumps(json.loads(add['stats']), indent=2)[:800])

{
  "numRecords": 500000,
  "minValues": {
    "do_zone": 1,
    "trip_id": 1,
    "pu_zone": 1,
    "pickup_at": "2024-01-01 00:00:00",
    "trip_distance": -0.0,
    "total_amount": -426.54,
    "passenger_count": 0
  },
  "maxValues": {
    "trip_distance": 59282.45,
    "pickup_at": "2024-01-06 16:38:19",
    "do_zone": 265,
    "passenger_count": 8,
    "pu_zone": 265,
    "trip_id": 500000,
    "total_amount": 1617.5
  },
  "nullCount": {
    "trip_id": 0,
    "do_zone": 0,
    "trip_distance": 0,
    "total_amount": 0,
    "passenger_count": 22098,
    "pu_zone": 0,
    "pickup_at": 0
  }
}


Nhớ Phase 1 chứ? Ta phải mở **footer của từng file Parquet** mới biết min/max để engine
bỏ qua row group. Bây giờ Delta đã **chép sẵn thống kê đó lên log**.

Khác biệt lớn hơn ta tưởng: một bảng 10.000 file thì cách Phase 1 phải mở 10.000 footer
(10.000 request tới object storage) chỉ để *lập kế hoạch* truy vấn. Với Delta, engine
đọc **một file log** là biết nên chạm vào file nào. Đó là **data skipping ở tầng bảng** —
và là một trong những lý do chính Delta nhanh trên dữ liệu lớn.

---
## Bước 3 — ACID: Delta khoá thế nào khi object storage không có khoá?

Ta ghi thêm một mẻ dữ liệu và soi xem log thay đổi ra sao.

In [5]:
# Mẻ tiếp theo: 100.000 chuyến kế tiếp trong tháng (mô phỏng job hôm sau chạy)
them = con.sql(f"""
    SELECT row_number() OVER (ORDER BY tpep_pickup_datetime) + 500000 AS trip_id,
           tpep_pickup_datetime AS pickup_at, PULocationID AS pu_zone,
           DOLocationID AS do_zone, passenger_count, trip_distance, total_amount
    FROM read_parquet('{RAW}')
    WHERE tpep_pickup_datetime >= '2024-01-01' AND tpep_pickup_datetime < '2024-02-01'
    ORDER BY tpep_pickup_datetime
    LIMIT 100000 OFFSET 500000
""").to_arrow_table()

write_deltalake(TABLE, them, mode='append', storage_options=SO)

dt = DeltaTable(TABLE, storage_options=SO)
print('Phiên bản hiện tại:', dt.version())
print('Số dòng           :', dt.to_pyarrow_dataset().count_rows())
print('Số file thuộc bảng:', len(dt.file_uris()))
doc_log(1)

Phiên bản hiện tại: 1
Số dòng           : 600000
Số file thuộc bảng: 2
━━━ phase2/bronze_trips/_delta_log/00000000000000000001.json ━━━

▸ action: commitInfo
{
  "timestamp": 1787482601162,
  "operation": "WRITE",
  "operationParameters": {
    "mode": "Append"
  },
  "engineInfo": "delta-rs:py-1.6.3",
  "operationMetrics": {
    "num_added_files": 1,
    "num_removed_files": 0,
    "num_partitions": 0,
    "num_added_rows": 100000,
    "execution_time_ms": 34
  },
  "clientVersion": "delta-rs.py-1.6.3"
}

▸ action: add
{
  "path": "part-00000-0f003970-f5ea-4e8d-9dc7-141e87601a75-c000.snappy.parquet",
  "partitionValues": {},
  "size": 1738028,
  "modificationTime": 1787482601162,
  "dataChange": true,
  "tags": null,
  "baseRowId": null,
  "defaultRowCommitVersion": null,
  "clusteringProvider": null
}


Commit số 1 **không hề nhắc tới file của commit 0**. Nó chỉ nói "thêm file này". Muốn
biết bảng gồm những file nào, engine đọc log từ đầu và **cộng dồn**: `add` thì thêm vào,
`remove` thì bớt ra. Cách làm này có tên: **log-structured**.

Chú ý action `commitInfo` — nó ghi lại *ai làm gì lúc nào*. Đó là mầm mống của
**audit và lineage**, thứ Phase 6 (Unity Catalog) sẽ khai thác đến nơi đến chốn.

### Giờ tới câu phỏng vấn khó nhất của Phase này

*"S3 không có khoá, không có transaction. Vậy Delta đảm bảo ACID kiểu gì khi mười job
cùng ghi một lúc?"*

Câu trả lời gói gọn trong **cách đặt tên file log**. Commit tiếp theo **bắt buộc** phải
tên là `...0002.json`. Hai writer cùng muốn commit sẽ cùng tranh tạo đúng một cái tên đó.
Nếu storage đảm bảo được rằng **chỉ một người tạo file mới thành công**, thì bài toán
xong: kẻ thắng được ghi, kẻ thua đọc lại log và thử lại với `...0003.json`.

Ta thử tận tay xem MinIO có đảm bảo được điều đó không.

In [6]:
# Ghi có điều kiện: "chỉ tạo nếu file CHƯA tồn tại" (HTTP header If-None-Match: *)
for lan in (1, 2):
    try:
        s3.put_object(Bucket=BUCKET, Key='phase2/thu_khoa.txt', Body=b'toi thang',
                      IfNoneMatch='*')
        print(f'Writer {lan}: ✓ commit THÀNH CÔNG')
    except Exception as e:
        print(f'Writer {lan}: ✗ bị chặn — {e.response["Error"]["Code"]}')

s3.delete_object(Bucket=BUCKET, Key='phase2/thu_khoa.txt')

Writer 1: ✓ commit THÀNH CÔNG
Writer 2: ✗ bị chặn — PreconditionFailed


{'ResponseMetadata': {'RequestId': '18CE6A16242DB6BB',
  'HostId': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
  'HTTPStatusCode': 204,
  'HTTPHeaders': {'accept-ranges': 'bytes',
   'server': 'MinIO',
   'strict-transport-security': 'max-age=31536000; includeSubDomains',
   'vary': 'Origin, Accept-Encoding',
   'x-amz-id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
   'x-amz-request-id': '18CE6A16242DB6BB',
   'x-content-type-options': 'nosniff',
   'x-ratelimit-limit': '2049',
   'x-ratelimit-remaining': '2049',
   'x-xss-protection': '1; mode=block',
   'date': 'Sun, 23 Aug 2026 10:59:05 GMT'},
  'RetryAttempts': 0}}

Đó chính là **toàn bộ cơ chế khoá của Delta**: một phép *put-if-absent* nguyên tử.
Không có khoá phân tán, không có ZooKeeper, không có tiến trình điều phối nào cả.

Ba hệ quả đáng nhớ cho phỏng vấn:

1. **Delta chỉ ACID được khi storage bên dưới cho phép "tạo file nếu chưa tồn tại".**
   Nó *mượn* tính nguyên tử của storage chứ không tự tạo ra.
2. **S3 thật mãi tới 2024 mới có tính năng này.** Trước đó Databricks phải dùng
   **DynamoDB làm trọng tài** cho nhiều writer trên S3 — đây là câu chuyện lịch sử rất
   hay ghi điểm khi được hỏi.
3. **Reader không bao giờ bị chặn.** Người đọc chỉ việc đọc snapshot ở phiên bản N.
   Writer ghi file mới hoàn toàn rồi mới commit — file chưa commit thì vô hình.
   Đây gọi là **snapshot isolation**, và nó chữa đúng vết thương cuối Phase 1:
   không ai còn nhìn thấy dữ liệu dở dang nữa.

In [7]:
# Chứng minh snapshot isolation: reader mở bảng TRƯỚC khi có ghi mới
reader = DeltaTable(TABLE, storage_options=SO)
truoc  = reader.to_pyarrow_dataset().count_rows()

write_deltalake(TABLE, them.slice(0, 1000), mode='append', storage_options=SO)   # writer chen ngang

print(f'Reader (đang giữ snapshot v{reader.version()}): {truoc:,} dòng — không đổi')
reader.update_incremental()                                       # tự nguyện xem bản mới
print(f'Reader sau khi làm mới  (v{reader.version()}): {reader.to_pyarrow_dataset().count_rows():,} dòng')

Reader (đang giữ snapshot v1): 600,000 dòng — không đổi
Reader sau khi làm mới  (v2): 601,000 dòng


---
## Bước 4 — Time travel: cái "Ctrl-Z" của dữ liệu

Log giữ lại **mọi phiên bản đã từng tồn tại**, nên đọc bảng ở quá khứ chỉ là việc
cộng dồn log tới commit thứ N rồi dừng.

In [8]:
dt = DeltaTable(TABLE, storage_options=SO)
lich_su = pd.DataFrame(dt.history())[['version', 'operation', 'timestamp']]
lich_su['luc'] = pd.to_datetime(lich_su['timestamp'], unit='ms')
lich_su[['version', 'operation', 'luc']]

,version,operation,luc
0,2,WRITE,2026-08-23 11:01:01.963
1,1,WRITE,2026-08-23 10:56:41.162
2,0,WRITE,2026-08-23 10:34:03.668


In [ ]:
for v in range(dt.version() + 1):
    n = DeltaTable(TABLE, version=v, storage_options=SO).to_pyarrow_dataset().count_rows()
    print(f'version {v}: {n:>9,} dòng')

### Time travel dùng để làm gì ngoài việc gây ấn tượng

- **Sửa sai.** Job chạy hỏng lúc 2 giờ sáng ghi đè dữ liệu bẩn → `RESTORE` về bản trước.
- **Tái lập thí nghiệm.** "Model này train trên dữ liệu ngày nào?" → chỉ vào version.
  Phase 10 (MLflow) sẽ dùng đúng chuyện này.
- **Kiểm toán.** So bảng hôm nay với bảng tuần trước, xem cái gì đã đổi.
- **Debug.** "Hôm qua báo cáo ra số khác" — giờ trả lời được chính xác vì sao.

In [9]:
# RESTORE: quay bảng về phiên bản 1. Chú ý — nó KHÔNG xoá lịch sử.
print(dt.restore(1))
dt = DeltaTable(TABLE, storage_options=SO)
print(f'\nSau restore: version {dt.version()}, {dt.to_pyarrow_dataset().count_rows():,} dòng')
print(pd.DataFrame(dt.history())[['version', 'operation']].head(3))

{'numRemovedFile': 1, 'numRestoredFile': 0}

Sau restore: version 3, 600,000 dòng
   version operation
0        3   RESTORE
1        2     WRITE
2        1     WRITE


Để ý điều tinh tế: `RESTORE` **không xoá** version 2. Nó tạo ra một commit *mới* nói
"bảng giờ gồm đúng các file của version 1". Lịch sử chỉ dài thêm, không bao giờ ngắn lại.

Đó là tính chất **append-only** của transaction log — cùng nguyên lý với sổ kế toán:
ghi sai thì ghi thêm bút toán điều chỉnh, không tẩy xoá dòng cũ.

---
## Bước 5 — `MERGE`: upsert, thứ mà data lake trần không làm nổi

Tình huống có thật, gặp hằng ngày:

- Dữ liệu **về muộn** (late-arriving): mấy chuyến hôm qua giờ mới tới.
- Dữ liệu **được sửa**: số tiền một chuyến bị tính sai, hệ thống nguồn gửi bản đúng.
- **CDC** từ database nguồn: một luồng gồm cả insert lẫn update (Phase 9 sẽ dựng đúng luồng này).

Cả ba đều cần một phép duy nhất: *"có rồi thì cập nhật, chưa có thì chèn"* — `MERGE`,
hay `UPSERT`. Với thư mục Parquet trần ở Phase 1, muốn sửa 2 dòng bạn phải đọc và ghi
lại toàn bộ bảng.

In [10]:
truoc_merge = con.sql(f"SELECT trip_id, total_amount FROM delta_scan('{TABLE}') WHERE trip_id IN (1, 2)").df()
print('Trước MERGE:'); print(truoc_merge)

# Nguồn: 2 dòng SỬA (trip_id 1,2) + 2 dòng MỚI (trip_id 999001,999002)
nguon = pa.table({
    'trip_id'         : pa.array([1, 2, 999001, 999002], pa.int64()),
    'pickup_at'       : pa.array(pd.to_datetime(['2024-01-01 00:00:01', '2024-01-01 00:00:02',
                                                 '2024-03-01 08:00:00', '2024-03-01 09:00:00'])),
    'pu_zone'         : pa.array([1, 2, 3, 4], pa.int32()),
    'do_zone'         : pa.array([1, 2, 3, 4], pa.int32()),
    'passenger_count' : pa.array([1, 1, 2, 2], pa.int64()),
    'trip_distance'   : pa.array([1.0, 2.0, 3.0, 4.0], pa.float64()),
    'total_amount'    : pa.array([999.99, 888.88, 50.0, 60.0], pa.float64()),
})

ket_qua = (DeltaTable(TABLE, storage_options=SO)
    .merge(nguon, predicate='t.trip_id = s.trip_id', source_alias='s', target_alias='t')
    .when_matched_update_all()
    .when_not_matched_insert_all()
    .execute())

print('\nKết quả MERGE:')
for k in ['num_source_rows', 'num_target_rows_updated', 'num_target_rows_inserted',
          'num_target_rows_copied', 'num_target_files_scanned',
          'num_target_files_removed', 'num_target_files_added']:
    print(f'  {k:28s}: {ket_qua[k]}')

Trước MERGE:
   trip_id  total_amount
0        1         11.25
1        2         16.32

Kết quả MERGE:
  num_source_rows             : 4
  num_target_rows_updated     : 2
  num_target_rows_inserted    : 2
  num_target_rows_copied      : 499998
  num_target_files_scanned    : 2
  num_target_files_removed    : 1
  num_target_files_added      : 1


In [11]:
sau_merge = con.sql(f"SELECT trip_id, total_amount FROM delta_scan('{TABLE}') WHERE trip_id IN (1, 2, 999001, 999002) ORDER BY trip_id").df()
print('Sau MERGE:'); print(sau_merge)

Sau MERGE:
   trip_id  total_amount
0        1        999.99
1        2        888.88
2   999001         50.00
3   999002         60.00


### Đọc kỹ mấy con số vừa in ra — có một sự thật quan trọng ẩn trong đó

`num_target_rows_copied` không phải 0. Delta **chép lại hàng trăm nghìn dòng không hề
thay đổi**, chỉ để sửa 2 dòng.

Vì sao? Vì file trên object storage **không sửa tại chỗ được**. Muốn đổi một dòng trong
file 50 MB, cách duy nhất là: đọc cả file → ghi ra file mới đã sửa → `remove` file cũ,
`add` file mới trong log. Cơ chế này tên là **copy-on-write**, và giá phải trả nằm ngay
trong hai con số `files_removed` / `files_added`.

Hệ quả thực dụng, rất hay được hỏi:

- **`MERGE` càng ít file bị chạm càng rẻ.** Nếu bảng partition theo ngày và bạn chỉ
  merge dữ liệu của một ngày, hãy đưa điều kiện đó vào `predicate` để Delta bỏ qua các
  partition khác. Predicate viết cẩu thả = quét cả bảng.
- **Delta có lối thoát khác: *deletion vectors*.** Thay vì ghi lại cả file, nó ghi một
  file phụ đánh dấu "dòng số 3, 7, 90 trong file kia đã chết". Gọi là
  **merge-on-read** — ghi nhanh hơn nhiều, đổi lại đọc phải chậm hơn chút. Đây đúng là
  đánh đổi kinh điển giữa *copy-on-write* và *merge-on-read*.

In [12]:
# MERGE trong log trông thế nào: một loạt remove + add trong CÙNG một commit
dt = DeltaTable(TABLE, storage_options=SO)
key  = f'{PREF}/_delta_log/{dt.version():020d}.json'
body = s3.get_object(Bucket=BUCKET, Key=key)['Body'].read().decode()

for line in body.strip().split('\n'):
    act  = json.loads(line)
    loai = list(act)[0]
    if loai in ('add', 'remove'):
        print(f'{loai:7s} {act[loai]["path"][:60]}')
    elif loai == 'commitInfo':
        print(f'commitInfo  operation = {act[loai].get("operation")}')

commitInfo  operation = MERGE
add     part-00000-8bfe5a3f-3328-4576-a0c4-4e6389c2e539-c000.snappy.
remove  part-00000-1643a0dc-3f75-457a-8d1b-7772d1c8e08e-c000.snappy.


Cả `remove` và `add` nằm trong **cùng một file JSON**. Đó là lý do người đọc không bao
giờ thấy trạng thái nửa vời: hoặc họ đọc log tới commit N-1 (thấy file cũ), hoặc tới
commit N (thấy file mới). **Không có khoảnh khắc ở giữa** — vì bản thân việc tạo file
commit là nguyên tử. Chữ **A** (atomicity) trong ACID nằm gọn ở đây.

---
## Bước 6 — Schema evolution

Sáu tháng nữa hệ thống nguồn thêm cột `tip_amount`. Ở data lake trần, ghi file có
schema khác vào cùng thư mục là công thức gây lỗi lúc đọc. Delta bắt bạn khai báo ý
định: mặc định nó **từ chối** — chỉ khi bạn nói rõ `schema_mode='merge'` mới cho qua.

In [13]:
moi = pa.table({
    'trip_id'         : pa.array([1000001], pa.int64()),
    'pickup_at'       : pa.array(pd.to_datetime(['2024-03-15 10:00:00'])),
    'pu_zone'         : pa.array([100], pa.int32()),
    'do_zone'         : pa.array([200], pa.int32()),
    'passenger_count' : pa.array([1], pa.int64()),
    'trip_distance'   : pa.array([5.5], pa.float64()),
    'total_amount'    : pa.array([25.0], pa.float64()),
    'tip_amount'      : pa.array([5.0], pa.float64()),      # ← cột mới
})

try:
    write_deltalake(TABLE, moi, mode='append', storage_options=SO)
except Exception as e:
    print('✗ Delta CHẶN lại:', str(e)[:300])

✗ Delta CHẶN lại: Cannot cast schema, number of fields does not match: 8 vs 7


In [14]:
write_deltalake(TABLE, moi, mode='append', schema_mode='merge', storage_options=SO)
dt = DeltaTable(TABLE, storage_options=SO)
print('Schema mới:', [f.name for f in dt.schema().fields])

# Dữ liệu cũ thì sao? Không file cũ nào bị ghi lại cả.
print(con.sql(f"""
    SELECT count(*) AS tong, count(tip_amount) AS co_tip
    FROM delta_scan('{TABLE}')
""").df())

Schema mới: ['trip_id', 'pickup_at', 'pu_zone', 'do_zone', 'passenger_count', 'trip_distance', 'total_amount', 'tip_amount']
     tong  co_tip
0  600003       1


Cột mới xuất hiện, dữ liệu cũ nhận `NULL` — **mà không file Parquet cũ nào bị ghi lại**.

Được thế là vì (nhớ bước 2): **schema nằm trong log, không nằm trong file dữ liệu**.
Thêm cột chỉ là ghi một action `metaData` mới. Khi đọc, engine thấy file cũ thiếu cột
`tip_amount` thì điền `NULL` — đúng chỗ này là nơi khái niệm *"schema-on-read"* của
data lake gặp *"schema có kiểm soát"* của warehouse, và **lakehouse** là tên gọi của
điểm gặp đó.

Đọc bảng ở phiên bản cũ sẽ thấy schema cũ — lịch sử của schema cũng được lưu vẹn nguyên.

---
## Bước 7 — `OPTIMIZE`: đúng việc bạn đã làm tay ở Phase 1

Ta tái hiện lại chính căn bệnh Phase 1: ghi 120 mẻ nhỏ (mô phỏng một job streaming
ghi vài phút một lần) vào bảng mới, đẻ ra một đống file vụn.

In [15]:
SMALL  = f's3://{BUCKET}/phase2/small_files'
PSMALL = 'phase2/small_files'

mau = con.sql(f"""
    SELECT row_number() OVER () AS trip_id, tpep_pickup_datetime AS pickup_at,
           PULocationID AS pu_zone, total_amount
    FROM read_parquet('{RAW}') WHERE tpep_pickup_datetime >= '2024-01-01' LIMIT 240000
""").to_arrow_table()

N_ME, CO_ME = 120, 2000
write_deltalake(SMALL, mau.slice(0, CO_ME), mode='overwrite', storage_options=SO)
for i in range(1, N_ME):                     # mỗi vòng lặp = một lần job ghi xuống lake
    write_deltalake(SMALL, mau.slice(i*CO_ME, CO_ME), mode='append', storage_options=SO)

dt_s = DeltaTable(SMALL, storage_options=SO)
truy_van = f"SELECT pu_zone, count(*) c, round(avg(total_amount),2) tb FROM delta_scan('{SMALL}') GROUP BY 1 ORDER BY c DESC LIMIT 5"

def do(sql, lan=3):
    return min([(lambda t0: (con.sql(sql).fetchall(), time.perf_counter()-t0)[1])(time.perf_counter()) for _ in range(lan)])

t_truoc = do(truy_van)
print(f'Số file : {len(dt_s.file_uris())}')
print(f'Version : {dt_s.version()}')
print(f'Query   : {t_truoc:.3f}s')

Số file : 120
Version : 119
Query   : 0.058s


In [16]:
kq = dt_s.optimize.compact()
print({k: kq[k] for k in ['numFilesAdded', 'numFilesRemoved', 'partitionsOptimized']})

dt_s = DeltaTable(SMALL, storage_options=SO)
t_sau = do(truy_van)
print(f'\nSố file sau OPTIMIZE : {len(dt_s.file_uris())}')
print(f'Query                : {t_sau:.3f}s   → nhanh hơn {t_truoc/t_sau:.1f} lần')

{'numFilesAdded': 1, 'numFilesRemoved': 120, 'partitionsOptimized': 1}

Số file sau OPTIMIZE : 1
Query                : 0.044s   → nhanh hơn 1.3 lần


In [17]:
def khoang_pu_zone(dt):
    """Mỗi file thuộc bảng trải trên khoảng pu_zone nào — đọc thẳng từ log."""
    a = pa.table(dt.get_add_actions(flatten=True)).to_pandas()
    return a[['path', 'num_records', 'min.pu_zone', 'max.pu_zone']].assign(
        path=lambda d: d['path'].str[11:19])   # rút gọn tên file cho dễ nhìn

# Trước Z-ORDER: file gộp chứa đủ mọi zone từ 1 tới 265 → thống kê vô dụng
print('Sau OPTIMIZE (chưa Z-ORDER):'); print(khoang_pu_zone(dt_s).to_string(index=False))

# Z-ORDER + ép kích thước file nhỏ để thấy rõ hiệu ứng gom cụm
dt_s.optimize.z_order(['pu_zone'], target_size=400_000)
dt_s = DeltaTable(SMALL, storage_options=SO)
print('\nSau Z-ORDER theo pu_zone:'); print(khoang_pu_zone(dt_s).to_string(index=False))

Sau OPTIMIZE (chưa Z-ORDER):
    path  num_records  min.pu_zone  max.pu_zone
3c968bb1       240000            1          265

Sau Z-ORDER theo pu_zone:
    path  num_records  min.pu_zone  max.pu_zone
5d5c31d7        17408            1           68
5d5c31d7        17408           68           90
5d5c31d7        18432           90          132
5d5c31d7        18432          132          132
5d5c31d7        17408          132          140
5d5c31d7        18432          140          144
5d5c31d7        18432          144          162
5d5c31d7        18432          162          164
5d5c31d7        18432          164          186
5d5c31d7        17408          186          230
5d5c31d7        18432          230          236
5d5c31d7        18432          236          239
5d5c31d7        18432          239          263
5d5c31d7         4480          263          265


Bảng min/max thứ hai chính là bài học bước 4 của Phase 1, lặp lại ở tầng cao hơn: sau
Z-ORDER mỗi file chỉ ôm một dải `pu_zone` hẹp, nên câu `WHERE pu_zone = 132` cho phép
engine **loại bỏ hầu hết file mà không mở chúng ra** — và lần này nó không cần mở
footer từng file, vì thống kê đã nằm sẵn trong log (bước 2).

(Trên bảng bé xíu chạy ở localhost, đồng hồ gần như không nhúc nhích — cả ở `compact`
lẫn `z_order`. Đừng tìm phép màu ở con số giây: bằng chứng nằm ở **số file** và ở
**bảng min/max**. Phase 1 đã cho bạn thấy cùng cơ chế ấy ở quy mô nghìn file thì đắt
tới mức nào — 14 lần.)

Ở Phase 1 bạn viết tay một câu `COPY ... ORDER BY ...` để gộp file. Hôm nay là
`optimize.compact()` và `optimize.z_order()`. **Cùng một việc** — nhưng khác nhau ở
điểm sống còn: Delta làm việc đó **trong một transaction**. Nghĩa là ai đang query giữa
chừng vẫn thấy bảng nguyên vẹn ở phiên bản cũ, cho tới đúng khoảnh khắc commit.

Đó là vết thương cuối cùng của Phase 1, giờ đã khép.

Chú ý cả hai lệnh đều **tạo commit mới** chứ không sửa lịch sử. File cũ vẫn nằm nguyên
trên storage, log chỉ đánh dấu chúng `remove`. Nghĩa là bảng của bạn hiện đang ôm cả
đống file không còn dùng nữa — và đó là lý do tồn tại của bước cuối.

---
## Bước 8 · SAI CÓ CHỦ ĐÍCH — `VACUUM` nuốt mất time travel

`VACUUM` xoá hẳn khỏi storage những file mà log đã đánh dấu `remove`. Nghe như dọn rác
vô hại. Nhưng hãy nghĩ kỹ một nhịp trước khi chạy cell dưới:

**Time travel về version cũ nghĩa là đọc những file cũ đó.** Xoá chúng đi thì...?

Ta thử trên một bảng riêng để không phá hỏng bảng chính.

In [18]:
DEMO  = f's3://{BUCKET}/phase2/demo_vacuum'
PDEMO = 'phase2/demo_vacuum'

write_deltalake(DEMO, pa.table({'id': [1, 2, 3]}), mode='overwrite', storage_options=SO)
write_deltalake(DEMO, pa.table({'id': [4, 5, 6]}), mode='overwrite', storage_options=SO)   # ghi đè

dt_d = DeltaTable(DEMO, storage_options=SO)
print('Bản hiện tại (v1):', dt_d.to_pandas()['id'].tolist())
print('Time travel (v0) :', DeltaTable(DEMO, version=0, storage_options=SO).to_pandas()['id'].tolist())
print('\nFile đang có trên storage:')
for o in s3.list_objects_v2(Bucket=BUCKET, Prefix=PDEMO)['Contents']:
    if o['Key'].endswith('.parquet'):
        print('  ', o['Key'].split('/')[-1])

Bản hiện tại (v1): [4, 5, 6]
Time travel (v0) : [1, 2, 3]

File đang có trên storage:
   part-00000-86a1b5f4-143c-4017-8470-88ab547ac88e-c000.snappy.parquet
   part-00000-dbf17c93-bc67-4f10-9eae-311fc134eb7c-c000.snappy.parquet


In [19]:
# retention_hours=0 + tắt kiểm tra an toàn = đúng cách người ta tự bắn vào chân mình
print('Sẽ xoá:', [f.split('/')[-1] for f in
      dt_d.vacuum(retention_hours=0, dry_run=True, enforce_retention_duration=False)])

dt_d.vacuum(retention_hours=0, dry_run=False, enforce_retention_duration=False)

try:
    print('Time travel v0:', DeltaTable(DEMO, version=0, storage_options=SO).to_pandas()['id'].tolist())
except Exception as e:
    print('✗ TIME TRAVEL CHẾT:', type(e).__name__, str(e)[:200])

Sẽ xoá: ['part-00000-86a1b5f4-143c-4017-8470-88ab547ac88e-c000.snappy.parquet']
✗ TIME TRAVEL CHẾT: FileNotFoundError Object at location phase2/demo_vacuum/part-00000-86a1b5f4-143c-4017-8470-88ab547ac88e-c000.snappy.parquet not found
                    ↳ Error performing GET http://minio:9000/lakehouse/phas


### Bài học đắt giá

Log vẫn còn nguyên — `dt.history()` vẫn kể được version 0 từng tồn tại. Nhưng **file dữ
liệu đã bốc hơi**, nên đọc quá khứ là lỗi. Lịch sử còn đó mà nội dung thì mất.

Vì thế Delta mặc định giữ **7 ngày** (`retention_hours=168`) và **cố tình chặn** nếu bạn
đặt ngắn hơn — chính là cái rào `enforce_retention_duration` mà ta vừa cố ý phá.

Ba điều rút ra:

1. **`VACUUM` = đánh đổi giữa tiền lưu trữ và độ dài quá khứ.** Không có lựa chọn miễn phí.
2. **Đừng bao giờ `VACUUM` với retention ngắn hơn job dài nhất đang chạy.** Một job Spark
   chạy 3 tiếng đang đọc snapshot cũ; bạn vacuum retention 1 tiếng ở giữa → job chết
   giữa chừng với `FileNotFoundException`. Đây là sự cố production **rất phổ biến**.
3. **Log và dữ liệu là hai thứ khác nhau.** `VACUUM` dọn file dữ liệu; dọn log là việc
   riêng (checkpoint + log retention).

In [20]:
# Cách làm đúng: để nguyên mặc định, và xem dry_run trước khi xoá thật
dt = DeltaTable(TABLE, storage_options=SO)
sap_xoa = dt.vacuum(retention_hours=168, dry_run=True)
print(f'Nếu vacuum bảng chính với retention 7 ngày, sẽ xoá {len(sap_xoa)} file')
print('(Bằng 0 là đúng — mọi file đều vừa được tạo trong vài phút qua.)')

Nếu vacuum bảng chính với retention 7 ngày, sẽ xoá 0 file
(Bằng 0 là đúng — mọi file đều vừa được tạo trong vài phút qua.)


---
## Bước 9 — Bằng chứng Delta là định dạng mở

Suốt notebook này ta ghi bằng `delta-rs` (Rust) nhưng phần lớn truy vấn lại chạy bằng
`delta_scan` của **DuckDB** — một engine hoàn toàn khác, do nhóm khác viết, không hề
biết `delta-rs` tồn tại.

Cả hai chỉ cần đồng ý với nhau về **một tập quy tắc đọc thư mục `_delta_log/`**.

In [21]:
print(con.sql(f"""
    SELECT CAST(pickup_at AS DATE) AS ngay,
           count(*) AS so_chuyen, round(avg(total_amount), 2) AS tb
    FROM delta_scan('{TABLE}')
    GROUP BY 1 ORDER BY 1 LIMIT 8
""").df())

print('\nĐọc bằng delta-rs (engine khác hẳn):',
      f'{DeltaTable(TABLE, storage_options=SO).to_pyarrow_dataset().count_rows():,} dòng')

        ngay  so_chuyen     tb
0 2024-01-01      81013  30.18
1 2024-01-02      75519  30.22
2 2024-01-03      82427  28.60
3 2024-01-04     102901  27.22
4 2024-01-05     103178  26.45
5 2024-01-06      97117  25.09
6 2024-01-07      57845  27.03
7 2024-03-01          2  55.00

Đọc bằng delta-rs (engine khác hẳn): 600,003 dòng


Ở Phase 3, **Spark** sẽ mở đúng bảng này. Phase 5, **Trino** cũng thế. Không ai phải
export/import gì cả — vì bảng chỉ là một thư mục trên MinIO theo chuẩn mở.

Đây chính là điều làm nên chữ *lakehouse*, và cũng là điều Databricks không thể khoá bạn lại.

---
## Tự kiểm tra — Phase 2 xong khi bạn làm được

**1. Vẽ lên giấy** cấu trúc một thư mục bảng Delta, chỉ ra chỗ nào là dữ liệu, chỗ nào
là metadata, và **bảng được xác định bởi cái gì**.

**2. Trả lời không cần tra cứu:**

- S3 không có khoá. Vậy hai job cùng ghi một bảng Delta thì ai thắng, và kẻ thua làm gì tiếp?
- Vì sao Databricks từng phải dùng DynamoDB cho Delta trên S3, còn bây giờ thì không?
- Thêm một cột vào bảng 10 TB mất bao lâu? Vì sao?
- `MERGE` sửa 2 dòng nhưng ghi lại hàng trăm nghìn dòng — vì sao, và *deletion vectors*
  đổi chuyện đó thế nào?
- Bạn `VACUUM` với retention 1 giờ. Chuyện gì xảy ra với job Spark đang chạy 3 tiếng?
- `RESTORE` về version 1 có xoá version 2 không? Vì sao câu trả lời lại quan trọng?

**3. Làm được bằng tay:** mở `_delta_log/` của một bảng lạ, chỉ ra bảng đó hiện gồm
những file nào **mà không cần chạy engine nào cả**.

**4. Nối vào bức tranh lớn:** Phase 1 để lại ba lỗ hổng (giao dịch, phiên bản, upsert) —
chỉ ra Delta vá từng lỗ bằng cơ chế nào. Rồi trả lời: **Delta vẫn còn thiếu gì?**
(Gợi ý: nó chạy trên một máy, dữ liệu vừa RAM. Đó là lý do Phase 3 tồn tại.)

> Xong mục 2 và 4 thì đánh dấu Phase 2 ✅ trong `docs/roadmap.md`.
> Phase 3 — Apache Spark — là phase nặng nhất cả lộ trình. Nghỉ một buổi rồi hẵng vào.